In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

FASTA_FILE = Path("../../0-download-inputs/data-files/orf_trans.fasta")

FILES = {
    "AE-MS": Path(
        "../../18-finalize/processed-data/"
        "20260210-s288c-ANnnotated-Yeast-Interactome.pkl"
    ),
    "Y2H": Path(
        "../../17-meltome-atlas/processed-data/y2h-test-s288c-step17.pkl"
    ),
    "Missing": Path(
        "../../17-meltome-atlas/processed-data/missing-s288c-step17.pkl"
    ),
}

OUTPUT_DIR = Path("processed-data")
SOURCE_COLUMN = "annotation_source_dataset"


# Read systematic identifiers from FASTA headers.
# Example: >YAL001C ...  ->  YAL001C
fasta_ids = []

with FASTA_FILE.open() as handle:
    for line in handle:
        if line.startswith(">"):
            header = line[1:].strip()
            if not header:
                raise ValueError("Found an empty FASTA header")
            fasta_ids.append(header.split()[0])

if not fasta_ids:
    raise ValueError(f"No FASTA records found: {FASTA_FILE}")

if len(fasta_ids) != len(set(fasta_ids)):
    raise ValueError("Duplicate identifiers found in the reference FASTA")

universe = set(fasta_ids)
print(f"Reference FASTA universe: {len(universe):,} proteins")


# Load annotated DataFrames.
frames = {}

for name, path in FILES.items():
    df = pd.read_pickle(path).copy()

    if not df.columns.is_unique or "node" not in df.columns:
        raise ValueError(f"{name}: missing 'node' or duplicate column names")

    if df["node"].isna().any():
        raise ValueError(f"{name}: null node identifiers")

    if not df["node"].map(lambda value: isinstance(value, str)).all():
        raise ValueError(f"{name}: non-string node identifiers")

    # Normalize surrounding whitespace without changing the source files.
    df["node"] = df["node"].str.strip()

    if df["node"].eq("").any():
        raise ValueError(f"{name}: blank node identifiers")

    if df["node"].duplicated().any():
        raise ValueError(f"{name}: duplicate node identifiers")

    if SOURCE_COLUMN in df.columns:
        raise ValueError(f"{name}: already contains {SOURCE_COLUMN!r}")

    frames[name] = df
    print(f"{name}: loaded {len(df):,} rows")


def expanded_membership(df):
    """Treat each identifier within a grouped node as represented."""
    return {
        identifier.strip()
        for node in df["node"]
        for identifier in node.split(";")
        if identifier.strip()
    }


network_members = {
    name: expanded_membership(frames[name])
    for name in ("AE-MS", "Y2H")
}

print("\nNETWORK MEMBERSHIP")
membership_summary = []

for name, members in network_members.items():
    membership_summary.append({
        "Network": name,
        "Expanded identifiers": len(members),
        "Identifiers in reference FASTA": len(members & universe),
        "Identifiers outside reference FASTA": len(members - universe),
    })

display(pd.DataFrame(membership_summary))


def build_missing_nodes(target, other):
    # Membership is determined exclusively from FASTA and the target network.
    expected_nodes = universe - network_members[target]

    # Use original Missing annotations first, then the other network.
    # The target file is a final fallback; its members cannot enter the output.
    annotation_tables = []

    for source in ("Missing", other, target):
        df = frames[source]

        # Exact individual identifiers only: never assign a grouped row's
        # annotations to every protein named within that row.
        selected = df.loc[df["node"].isin(expected_nodes)].copy()
        selected[SOURCE_COLUMN] = source
        annotation_tables.append(selected)

    annotations = pd.concat(
        annotation_tables,
        ignore_index=True,
        sort=False,
    )

    # Preserve one annotation row per node using the explicit priority above.
    annotations = annotations.drop_duplicates("node", keep="first")

    # Reindex to the complete FASTA-defined complement. This retains proteins
    # even if none of the supplied DataFrames contains their annotations.
    result = (
        annotations.set_index("node")
        .reindex(sorted(expected_nodes))
        .rename_axis("node")
        .reset_index()
    )

    actual_nodes = set(result["node"])

    if actual_nodes != expected_nodes:
        raise ValueError(f"{target}: output does not match the expected complement")

    if result["node"].duplicated().any():
        raise ValueError(f"{target}: duplicate output nodes")

    if actual_nodes & network_members[target]:
        raise ValueError(f"{target}: output overlaps the target network")

    unannotated = result.loc[result[SOURCE_COLUMN].isna(), "node"]

    print(
        f"\nMissing from {target}: {len(result):,} proteins; "
        f"{len(unannotated):,} without an annotation row"
    )

    if not unannotated.empty:
        print("Reference proteins requiring annotation review:")
        display(unannotated.to_frame())

    return result


missing_from_ae_ms = build_missing_nodes("AE-MS", "Y2H")
missing_from_y2h = build_missing_nodes("Y2H", "AE-MS")


# Save both formats after both membership checks pass.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

outputs = {
    "Missing from AE-MS": (
        missing_from_ae_ms,
        OUTPUT_DIR / "missing-from-ae-ms-s288c.pkl",
    ),
    "Missing from Y2H": (
        missing_from_y2h,
        OUTPUT_DIR / "missing-from-y2h-s288c.pkl",
    ),
}

for name, (df, path) in outputs.items():
    df.to_pickle(path)

    csv_path = path.with_suffix(".csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"\n{name}: {len(df):,} rows")
    print(f"  Wrote {path}")
    print(f"  Wrote {csv_path}")

Reference FASTA universe: 6,039 proteins
AE-MS: loaded 3,927 rows
Y2H: loaded 2,018 rows
Missing: loaded 1,622 rows

NETWORK MEMBERSHIP


,Network,Expanded identifiers,Identifiers in reference FASTA,Identifiers outside reference FASTA
0,AE-MS,3939,3928,11
1,Y2H,2018,1960,58



Missing from AE-MS: 2,111 proteins; 0 without an annotation row

Missing from Y2H: 4,079 proteins; 44 without an annotation row
Reference proteins requiring annotation review:


,node
69,YAR009C
148,YBL100W-A
525,YDL083C
627,YDR025W
634,YDR034C-C
635,YDR034C-D
639,YDR038C
640,YDR039C
809,YDR316W-B
945,YDR545W



Missing from AE-MS: 2,111 rows
  Wrote processed-data/missing-from-ae-ms-s288c.pkl
  Wrote processed-data/missing-from-ae-ms-s288c.csv

Missing from Y2H: 4,079 rows
  Wrote processed-data/missing-from-y2h-s288c.pkl
  Wrote processed-data/missing-from-y2h-s288c.csv


In [3]:
nodes = frames["AE-MS"]["node"]

whole_node_ids = set(nodes)
expanded_ids = expanded_membership(frames["AE-MS"])

whole_reference_ids = whole_node_ids & universe
expanded_reference_ids = expanded_ids & universe

print(f"Network rows:                         {len(nodes):,}")
print(f"Unique whole node labels:             {len(whole_node_ids):,}")
print(f"Rows containing semicolons:           {nodes.str.contains(';', regex=False).sum():,}")
print(f"Whole labels matching FASTA IDs:      {len(whole_reference_ids):,}")
print(f"Reference IDs after splitting:        {len(expanded_reference_ids):,}")
print(
    "Additional reference IDs exposed by splitting: "
    f"{len(expanded_reference_ids - whole_reference_ids):,}"
)

# Show the grouped rows and the reference loci they represent.
grouped_rows = []

for label in nodes[nodes.str.contains(";", regex=False)]:
    tokens = {token.strip() for token in label.split(";") if token.strip()}
    grouped_rows.append({
        "network_node": label,
        "reference_loci": ";".join(sorted(tokens & universe)),
        "other_identifiers": ";".join(sorted(tokens - universe)),
        "number_of_reference_loci": len(tokens & universe),
    })

display(pd.DataFrame(grouped_rows))

Network rows:                         3,927
Unique whole node labels:             3,927
Rows containing semicolons:           41
Whole labels matching FASTA IDs:      3,876
Reference IDs after splitting:        3,928
Additional reference IDs exposed by splitting: 52


,network_node,reference_loci,other_identifiers,number_of_reference_loci
0,YGR027W-A;YGR038C-A;YJR026W;YLR227W-A;YML040W,YGR027W-A;YGR038C-A;YJR026W;YLR227W-A;YML040W,,5
1,YJR145C;YHR203C,YHR203C;YJR145C,,2
2,YOR390W;YPL279C,YOR390W;YPL279C,,2
3,YLL039C;YLR167W,YLL039C;YLR167W,,2
4,YBL072C;YER102W,YBL072C;YER102W,,2
5,YDR025W;YBR048W,YBR048W;YDR025W,,2
6,YER074W;YIL069C,YER074W;YIL069C,,2
7,YDR450W;YML026C,YDR450W;YML026C,,2
8,YGR118W;YPR132W,YGR118W;YPR132W,,2
9,YJL190C;YLR367W,YJL190C;YLR367W,,2
